# 03 — Método dos Mínimos Quadrados (MMQ)

**Métodos Numéricos — Engenharia Naval (Poli-USP)**

Neste notebook vamos estudar o **Método dos Mínimos Quadrados (MMQ)**, com foco em duas aplicações centrais da disciplina:

- **Ajuste de curvas** (regressão) a dados experimentais com ruído;
- **Aproximação por Polinômios de Legendre** (base ortogonal de funções);
- **Análise Harmônica** (Séries de Fourier), que é um caso particular de MMQ usando senos e cossenos como base.

A ideia central do MMQ é: dado um conjunto de dados (ou uma função) que **não pode (ou não deve) ser interpolado exatamente**, buscamos a curva de uma família escolhida (polinômios, harmônicos, etc.) que **minimiza a soma dos quadrados dos resíduos** (erros) entre o modelo e os dados/função.

> Este notebook é autossuficiente: basta executar as células em ordem, de cima para baixo.

## Formulário

As fórmulas abaixo são as que constam no formulário oficial da disciplina (`formulário.pdf`) para os tópicos de Polinômios de Legendre e Análise Harmônica. Vamos usá-las como referência ao longo do notebook.

### Polinômios de Legendre

$$p_0(x) = 1, \quad p_1(x) = x, \quad p_2(x) = \frac{3x^2-1}{2}, \quad p_3(x) = \frac{5x^3-3x}{2}, \quad \dots$$

Estes polinômios satisfazem (ortogonalidade):

$$\int_{-1}^{1} p_i(x)\, p_j(x)\, dx = \begin{cases} 0 & \text{se } i \neq j \\[4pt] \dfrac{2}{2j+1} & \text{se } i = j \end{cases}$$

### Análise Harmônica

$$f(x) = a_0 + \sum_{n=1}^{\infty} \left( a_n \cos\frac{n\pi x}{L} + b_n \sin\frac{n\pi x}{L} \right)$$

onde

$$a_0 = \frac{1}{2L}\int_{-L}^{L} f(x)\,dx, \qquad a_n = \frac{1}{L}\int_{-L}^{L} f(x)\cos\frac{n\pi x}{L}\,dx, \qquad b_n = \frac{1}{L}\int_{-L}^{L} f(x)\sin\frac{n\pi x}{L}\,dx$$

In [ ]:
# Bibliotecas usadas no notebook
import numpy as np
import matplotlib.pyplot as plt

# Semente fixa para reprodutibilidade dos dados com ruido
np.random.seed(42)

# Paleta de cores consistente usada em todos os graficos
COR_DADOS = '#1f77b4'      # pontos experimentais
COR_AJUSTE = '#d62728'     # curva ajustada / modelo
COR_RESIDUO = '#7f7f7f'    # residuos
COR_REF = '#2ca02c'        # funcao de referencia / exata

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Compatibilidade: np.trapz foi renomeado para np.trapezoid no NumPy >= 2.0
trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

## 1. Motivação: por que não interpolar?

Quando temos **dados experimentais** (por exemplo, medições de um ensaio em tanque de provas, ou de um sensor em um navio), eles sempre vêm acompanhados de **ruído de medição**. Se tentarmos *interpolar* esses dados (fazer uma curva passar exatamente por cada ponto), estaremos reproduzindo também o ruído — o que geralmente é indesejado e gera curvas com oscilações artificiais (lembra do fenômeno de Runge, visto com interpolação polinomial).

O **Método dos Mínimos Quadrados (MMQ)** resolve esse problema de outra forma: em vez de exigir que a curva passe exatamente pelos pontos, escolhemos uma **família de funções simples** (ex.: polinômios de grau baixo, ou combinações de senos/cossenos) e buscamos os parâmetros que **minimizam a soma dos quadrados dos erros (resíduos)**:

$$S = \sum_{i=1}^{m} \big(y_i - f(x_i)\big)^2 \;\longrightarrow\; \text{mínimo}$$

onde $y_i$ são os dados observados e $f(x_i)$ é o valor previsto pelo modelo ajustado.

### Interpretação geométrica

Uma forma elegante (e muito usada em Métodos Numéricos) de pensar o MMQ é como uma **projeção ortogonal** no espaço de funções (ou de vetores):

- Os dados $y$ formam um vetor em um espaço de dimensão $m$ (número de pontos);
- O modelo $f(x) = c_0 \phi_0(x) + c_1 \phi_1(x) + \dots + c_n \phi_n(x)$ (combinação linear de funções-base $\phi_j$) define um **subespaço** de dimensão $n+1$ dentro desse espaço;
- O MMQ escolhe exatamente o ponto desse subespaço **mais próximo** do vetor de dados $y$, na métrica euclidiana — ou seja, a **projeção ortogonal** de $y$ sobre o subespaço gerado pelas funções-base.

Essa é a razão pela qual o resíduo ótimo $r = y - f(x)$ é **ortogonal** a todas as funções da base — é exatamente essa condição de ortogonalidade que gera o chamado **sistema de equações normais**, que veremos a seguir.

## 2. Ajuste polinomial pelo sistema de equações normais

Vamos ajustar um polinômio $f(x) = c_0 + c_1 x + c_2 x^2 + \dots + c_n x^n$ a um conjunto de $m$ pontos $(x_i, y_i)$, com $m > n+1$ (mais pontos do que parâmetros — sistema sobredeterminado).

Definindo a **matriz de projeto** $A \in \mathbb{R}^{m \times (n+1)}$, com $A_{ij} = x_i^{\,j}$, o modelo em forma matricial é $Ac \approx y$. Minimizar $\lVert y - Ac \rVert^2$ leva ao **sistema de equações normais**:

$$A^T A\, c = A^T y$$

que é um sistema linear de $(n+1)$ equações e $(n+1)$ incógnitas, resolvido diretamente (por eliminação de Gauss, por exemplo).

Vamos implementar isso **do zero** (sem `numpy.polyfit`) e testar em dados sintéticos ruidosos.

In [ ]:
# Gera dados sinteticos: y = sin(x) + ruido gaussiano
m = 40                                   # numero de pontos (amostras)
x_dados = np.linspace(0, 2 * np.pi, m)
ruido = np.random.normal(loc=0.0, scale=0.15, size=m)
y_dados = np.sin(x_dados) + ruido

plt.figure()
plt.plot(x_dados, np.sin(x_dados), color=COR_REF, linestyle='--', label=r'$\sin(x)$ (funcao geradora, sem ruido)')
plt.scatter(x_dados, y_dados, color=COR_DADOS, zorder=3, label='dados com ruido')
plt.title(r'Dados sinteticos: $y = \sin(x) + \varepsilon$')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
def matriz_vandermonde(x, grau):
    """Monta a matriz de projeto A (tipo Vandermonde) para o ajuste polinomial.

    A[i, j] = x_i ** j,  j = 0, 1, ..., grau
    """
    x = np.asarray(x, dtype=float)
    m = len(x)
    A = np.zeros((m, grau + 1))
    for j in range(grau + 1):
        A[:, j] = x ** j
    return A


def resolve_sistema_normal(A, y):
    """Resolve o sistema de equacoes normais (A^T A) c = A^T y.

    Usamos np.linalg.solve apenas para resolver o sistema linear ja
    montado -- a parte conceitual do MMQ (montar A^T A e A^T y) e feita
    manualmente, sem usar numpy.polyfit ou numpy.lstsq.
    """
    ATA = A.T @ A
    ATy = A.T @ y
    c = np.linalg.solve(ATA, ATy)
    return c


# Ajuste polinomial de grau 3 aos dados ruidosos
grau = 3
A = matriz_vandermonde(x_dados, grau)
coef = resolve_sistema_normal(A, y_dados)

print(f'Coeficientes do polinomio ajustado (grau {grau}):')
for j, cj in enumerate(coef):
    print(f'  c{j} = {cj: .5f}')

# Avalia o polinomio ajustado numa malha fina para plotar a curva suave
x_fino = np.linspace(x_dados.min(), x_dados.max(), 300)
A_fino = matriz_vandermonde(x_fino, grau)
y_ajuste_fino = A_fino @ coef

# Avalia tambem nos proprios pontos dos dados, para calcular os residuos
y_ajuste_dados = A @ coef
residuos = y_dados - y_ajuste_dados
SQR = np.sum(residuos ** 2)
print(f'\nSoma dos quadrados dos residuos (SQR) = {SQR:.5f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# --- Painel de cima: dados, curva ajustada e residuos (linhas verticais) ---
ax1.scatter(x_dados, y_dados, color=COR_DADOS, zorder=3, label='dados com ruido')
ax1.plot(x_fino, y_ajuste_fino, color=COR_AJUSTE, linewidth=2,
          label=f'polinomio ajustado (grau {grau})')
ax1.plot(x_dados, np.sin(x_dados), color=COR_REF, linestyle='--', alpha=0.7,
          label=r'$\sin(x)$ (referencia)')

# desenha os residuos como segmentos verticais entre o dado e a curva ajustada
for xi, yi, yfi in zip(x_dados, y_dados, y_ajuste_dados):
    ax1.plot([xi, xi], [yi, yfi], color=COR_RESIDUO, linewidth=1, alpha=0.6, zorder=2)

ax1.set_ylabel('y')
ax1.set_title('Ajuste por Minimos Quadrados (sistema normal) e residuos')
ax1.legend()

# --- Painel de baixo: residuos isolados ---
ax2.axhline(0, color='black', linewidth=0.8)
ax2.stem(x_dados, residuos, linefmt=COR_RESIDUO, markerfmt='o', basefmt=' ')
ax2.set_xlabel('x')
ax2.set_ylabel('residuo')
ax2.set_title('Residuos $y_i - f(x_i)$')

plt.tight_layout()
plt.show()

## 3. Base de funções ortogonais: Polinômios de Legendre

No exemplo anterior usamos a **base monomial** $\{1, x, x^2, x^3, \dots\}$ para montar a matriz $A$. Isso funciona, mas tem um problema sério: para graus altos, as colunas de $A$ (e portanto da matriz normal $A^T A$) ficam **quase paralelas** entre si (pois $x^k$ e $x^{k+1}$ são parecidas no intervalo de interesse), o que deixa $A^T A$ **mal condicionada**. Pequenos erros de arredondamento podem gerar coeficientes completamente errados.

A solução clássica é usar uma **base ortogonal** de polinômios, como os **Polinômios de Legendre** $p_0, p_1, p_2, \dots$, que satisfazem a propriedade de ortogonalidade mostrada no Formulário:

$$\int_{-1}^{1} p_i(x)\, p_j(x)\, dx = 0 \quad \text{se } i \neq j$$

Quando a base é ortogonal, a matriz $A^T A$ (ou a matriz de Gram, no caso contínuo) é **diagonal** — o sistema normal se desacopla completamente, e cada coeficiente pode ser calculado **de forma independente** por uma simples projeção, sem necessidade de resolver um sistema linear (e sem o problema de mau condicionamento).

### Fórmula de recorrência

Os polinômios de Legendre também podem ser gerados pela relação de recorrência de três termos:

$$(n+1)\, p_{n+1}(x) = (2n+1)\, x\, p_n(x) - n\, p_{n-1}(x)$$

com $p_0(x) = 1$ e $p_1(x) = x$. Vamos implementar essa recorrência.

In [ ]:
def legendre(n, x):
    """Calcula o polinomio de Legendre p_n avaliado em x (escalar ou array),
    usando a recorrencia de tres termos:

        (n+1) p_{n+1}(x) = (2n+1) x p_n(x) - n p_{n-1}(x)

    com p_0(x) = 1 e p_1(x) = x.
    """
    x = np.asarray(x, dtype=float)
    p_anterior = np.ones_like(x)   # p_0
    if n == 0:
        return p_anterior
    p_atual = x.copy()             # p_1
    if n == 1:
        return p_atual

    for k in range(1, n):
        p_proximo = ((2 * k + 1) * x * p_atual - k * p_anterior) / (k + 1)
        p_anterior, p_atual = p_atual, p_proximo
    return p_atual


# Verificacao rapida contra as formulas explicitas do formulario
x_teste = np.linspace(-1, 1, 5)
print('p2 (recorrencia):', legendre(2, x_teste))
print('p2 (formula)     :', (3 * x_teste**2 - 1) / 2)
print('p3 (recorrencia):', legendre(3, x_teste))
print('p3 (formula)     :', (5 * x_teste**3 - 3 * x_teste) / 2)

In [ ]:
# Visualiza os primeiros polinomios de Legendre p0..p4 em [-1, 1]
x_leg = np.linspace(-1, 1, 400)
cores_legendre = plt.cm.viridis(np.linspace(0, 0.9, 5))

plt.figure()
for n in range(5):
    plt.plot(x_leg, legendre(n, x_leg), color=cores_legendre[n], linewidth=2, label=f'$p_{n}(x)$')

plt.axhline(0, color='black', linewidth=0.6)
plt.axvline(0, color='black', linewidth=0.6)
plt.title('Polinomios de Legendre $p_0, \\dots, p_4$')
plt.xlabel('x')
plt.ylabel('$p_n(x)$')
plt.legend()
plt.show()

### Verificando a ortogonalidade numericamente

Vamos calcular numericamente as integrais $\int_{-1}^{1} p_i(x) p_j(x)\,dx$ para $i,j = 0,\dots,4$ e verificar que a matriz resultante é (aproximadamente) diagonal, com os valores $\frac{2}{2j+1}$ na diagonal, conforme o Formulário.

In [ ]:
# Malha fina para integracao numerica (regra do trapezio)
x_int = np.linspace(-1, 1, 2001)

N = 5  # testamos p0..p4
produtos_internos = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        integrando = legendre(i, x_int) * legendre(j, x_int)
        produtos_internos[i, j] = trapz(integrando, x_int)

print('Matriz de produtos internos <p_i, p_j> (deve ser ~diagonal):')
with np.printoptions(precision=4, suppress=True):
    print(produtos_internos)

print('\nValores esperados na diagonal (2/(2j+1)):')
print([2 / (2 * j + 1) for j in range(N)])

# Heatmap da matriz de produtos internos
plt.figure(figsize=(5.5, 5))
im = plt.imshow(produtos_internos, cmap='viridis')
plt.colorbar(im, label=r'$\int_{-1}^{1} p_i p_j\, dx$')
plt.xticks(range(N), [f'$p_{j}$' for j in range(N)])
plt.yticks(range(N), [f'$p_{i}$' for i in range(N)])
plt.title('Matriz de produtos internos (ortogonalidade de Legendre)')

# anota os valores numericos em cada celula
for i in range(N):
    for j in range(N):
        plt.text(j, i, f'{produtos_internos[i, j]:.2f}', ha='center', va='center',
                  color='white' if produtos_internos[i, j] < produtos_internos.max() / 2 else 'black')
plt.show()

### Ajuste por projeção na base de Legendre

Como a base é ortogonal, não precisamos resolver nenhum sistema linear: cada coeficiente é obtido por uma **projeção direta** da função sobre $p_j$:

$$c_j = \frac{2j+1}{2}\int_{-1}^{1} f(x)\, p_j(x)\, dx$$

Vamos aproximar $f(x) = e^x$ no intervalo $[-1, 1]$ usando poucos polinômios de Legendre e comparar com a função exata.

In [ ]:
def f_alvo(x):
    """Funcao que queremos aproximar no intervalo [-1, 1]."""
    return np.exp(x)


def coeficiente_legendre(j, f, x_int):
    """Calcula c_j = (2j+1)/2 * integral de f(x) p_j(x) dx em [-1, 1],
    por integracao numerica (regra do trapezio).
    """
    integrando = f(x_int) * legendre(j, x_int)
    return (2 * j + 1) / 2 * trapz(integrando, x_int)


grau_legendre = 4  # usamos p0..p4 (5 termos)
coefs_legendre = [coeficiente_legendre(j, f_alvo, x_int) for j in range(grau_legendre + 1)]
print('Coeficientes de Legendre de f(x) = exp(x):')
for j, cj in enumerate(coefs_legendre):
    print(f'  c{j} = {cj: .5f}')

# Reconstroi a aproximacao somando os termos c_j * p_j(x)
x_plot = np.linspace(-1, 1, 400)
aproximacao = sum(cj * legendre(j, x_plot) for j, cj in enumerate(coefs_legendre))

plt.figure()
plt.plot(x_plot, f_alvo(x_plot), color=COR_REF, linewidth=2, label='$f(x) = e^x$ (exata)')
plt.plot(x_plot, aproximacao, color=COR_AJUSTE, linestyle='--', linewidth=2,
         label=f'aproximacao com p0..p{grau_legendre}')
plt.title('Aproximacao de $e^x$ por projecao em polinomios de Legendre')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

### Condicionamento: base monomial vs. base de Legendre

Para ilustrar por que a base ortogonal é preferível, vamos montar a **matriz de Gram** (produtos internos entre as funções da base, calculados numericamente em $[-1,1]$) tanto para a base monomial $\{1, x, x^2, \dots\}$ quanto para a base de Legendre $\{p_0, p_1, p_2, \dots\}$, e comparar o **número de condição** (`numpy.linalg.cond`) de cada uma. Quanto maior o número de condição, mais sensível (instável) é a solução do sistema normal a pequenos erros numéricos.

In [ ]:
def matriz_gram(funcoes, x_int):
    """Monta a matriz de Gram G_ij = integral de f_i(x) f_j(x) dx em [-1, 1],
    para uma lista de funcoes-base, usando integracao numerica.
    """
    k = len(funcoes)
    G = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            G[i, j] = trapz(funcoes[i](x_int) * funcoes[j](x_int), x_int)
    return G


graus_testados = [2, 4, 6, 8, 10]
cond_monomial = []
cond_legendre = []

for grau in graus_testados:
    funcoes_monomiais = [(lambda x, j=j: x ** j) for j in range(grau + 1)]
    funcoes_legendre = [(lambda x, j=j: legendre(j, x)) for j in range(grau + 1)]

    G_mono = matriz_gram(funcoes_monomiais, x_int)
    G_leg = matriz_gram(funcoes_legendre, x_int)

    cond_monomial.append(np.linalg.cond(G_mono))
    cond_legendre.append(np.linalg.cond(G_leg))

print(f'{"grau":>6} | {"cond. monomial":>16} | {"cond. Legendre":>16}')
for grau, cm, cl in zip(graus_testados, cond_monomial, cond_legendre):
    print(f'{grau:>6} | {cm:>16.3e} | {cl:>16.3e}')

plt.figure()
plt.semilogy(graus_testados, cond_monomial, 'o-', color=COR_AJUSTE, label='base monomial')
plt.semilogy(graus_testados, cond_legendre, 's-', color=COR_REF, label='base de Legendre')
plt.xlabel('grau do polinomio (numero de termos - 1)')
plt.ylabel('numero de condicao (escala log)')
plt.title('Condicionamento da matriz de Gram: monomial vs. Legendre')
plt.legend()
plt.show()

## 4. Análise Harmônica (Séries de Fourier)

A **Análise Harmônica** é, na verdade, mais um caso de MMQ: aproximamos uma função periódica $f(x)$, de período $2L$, por uma combinação linear de funções **senos e cossenos** (que também formam uma base ortogonal, assim como os polinômios de Legendre):

$$f(x) \approx a_0 + \sum_{n=1}^{N} \left( a_n \cos\frac{n\pi x}{L} + b_n \sin\frac{n\pi x}{L} \right)$$

Os coeficientes (ver Formulário) são obtidos por projeção (integrais):

$$a_0 = \frac{1}{2L}\int_{-L}^{L} f(x)\,dx, \qquad a_n = \frac{1}{L}\int_{-L}^{L} f(x)\cos\frac{n\pi x}{L}\,dx, \qquad b_n = \frac{1}{L}\int_{-L}^{L} f(x)\sin\frac{n\pi x}{L}\,dx$$

Vamos calcular esses coeficientes **numericamente** (integração pela regra do trapézio, com `numpy.trapz`) para duas funções de teste:

1. A **onda dente de serra**, $f(x) = x$ em $(-L, L)$ (contínua por partes, mas com uma descontinuidade de salto a cada período);
2. A **onda quadrada** (descontínua), para observarmos o **fenômeno de Gibbs**.

In [ ]:
L = np.pi  # semi-periodo: f tem periodo 2L


def onda_dente_de_serra(x):
    """f(x) = x no intervalo (-L, L), estendida periodicamente."""
    x_mod = np.mod(x + L, 2 * L) - L  # traz x para o intervalo (-L, L]
    return x_mod


def onda_quadrada(x):
    """Onda quadrada: +1 em (0, L), -1 em (-L, 0), periodica com periodo 2L."""
    x_mod = np.mod(x + L, 2 * L) - L
    return np.where(x_mod >= 0, 1.0, -1.0)


def coeficientes_fourier(f, L, N, n_pontos=4001):
    """Calcula a0, [a1..aN], [b1..bN] por integracao numerica (trapezio)
    no intervalo [-L, L].
    """
    x_int = np.linspace(-L, L, n_pontos)
    fx = f(x_int)

    a0 = (1 / (2 * L)) * trapz(fx, x_int)

    a = np.zeros(N + 1)  # a[n], usamos indices 1..N (a[0] fica sem uso)
    b = np.zeros(N + 1)
    for n in range(1, N + 1):
        a[n] = (1 / L) * trapz(fx * np.cos(n * np.pi * x_int / L), x_int)
        b[n] = (1 / L) * trapz(fx * np.sin(n * np.pi * x_int / L), x_int)

    return a0, a, b


def reconstroi_serie(x, a0, a, b, L, N):
    """Reconstroi a serie de Fourier truncada em N termos, avaliada em x."""
    x = np.asarray(x, dtype=float)
    soma = a0 * np.ones_like(x)
    for n in range(1, N + 1):
        soma = soma + a[n] * np.cos(n * np.pi * x / L) + b[n] * np.sin(n * np.pi * x / L)
    return soma


# Calcula os coeficientes (ate N=50) para as duas funcoes de teste
N_MAX = 50
a0_serra, a_serra, b_serra = coeficientes_fourier(onda_dente_de_serra, L, N_MAX)
a0_quad, a_quad, b_quad = coeficientes_fourier(onda_quadrada, L, N_MAX)

print(f'Onda dente de serra: a0 = {a0_serra:.5f} (esperado ~0)')
print(f'Onda quadrada      : a0 = {a0_quad:.5f} (esperado ~0)')

In [ ]:
# (a) Visualiza as duas funcoes de teste, em varios periodos
x_vis = np.linspace(-3 * L, 3 * L, 2000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_vis, onda_dente_de_serra(x_vis), color=COR_REF)
axes[0].set_title('Onda dente de serra: $f(x) = x$ em $(-L, L)$')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')

axes[1].plot(x_vis, onda_quadrada(x_vis), color=COR_AJUSTE)
axes[1].set_title('Onda quadrada (descontinua)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')

plt.tight_layout()
plt.show()

In [ ]:
# (b) Convergencia da serie truncada para a onda dente de serra
x_rec = np.linspace(-L, L, 1000)
Ns = [1, 3, 5, 10, 50]
cores_N = plt.cm.plasma(np.linspace(0.1, 0.85, len(Ns)))

plt.figure(figsize=(9, 5.5))
plt.plot(x_rec, onda_dente_de_serra(x_rec), color='black', linewidth=2.5, label='f(x) exata', zorder=5)
for N, cor in zip(Ns, cores_N):
    y_rec = reconstroi_serie(x_rec, a0_serra, a_serra, b_serra, L, N)
    plt.plot(x_rec, y_rec, color=cor, linewidth=1.5, label=f'N = {N}')

plt.title('Convergencia da serie de Fourier - onda dente de serra')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.show()

### (c) O fenômeno de Gibbs

Quando a função tem uma **descontinuidade de salto** (como a onda quadrada), a série de Fourier truncada **sempre apresenta um "overshoot"** (ultrapassagem) perto da descontinuidade — e, surpreendentemente, esse overshoot **não desaparece** quando aumentamos o número de termos $N$! Ele apenas fica mais estreito (concentrado perto do salto), mas sua altura tende a um valor constante (~9% do salto). Isso é conhecido como **fenômeno de Gibbs**.

In [ ]:
# Fenomeno de Gibbs: reconstrucao da onda quadrada com N crescente,
# com zoom perto da descontinuidade em x = 0
Ns_gibbs = [5, 15, 50, 150]
cores_gibbs = plt.cm.plasma(np.linspace(0.1, 0.85, len(Ns_gibbs)))

a0_quad, a_quad, b_quad = coeficientes_fourier(onda_quadrada, L, max(Ns_gibbs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for ax, titulo in [(ax1, 'visao geral'), (ax2, 'zoom perto do salto em x=0')]:
    ax.plot(x_rec, onda_quadrada(x_rec), color='black', linewidth=2.5, label='f(x) exata', zorder=5)
    for N, cor in zip(Ns_gibbs, cores_gibbs):
        y_rec = reconstroi_serie(x_rec, a0_quad, a_quad, b_quad, L, N)
        ax.plot(x_rec, y_rec, color=cor, linewidth=1.3, label=f'N = {N}')
    ax.axhline(1.0, color='gray', linestyle=':', linewidth=0.8)
    ax.axhline(-1.0, color='gray', linestyle=':', linewidth=0.8)
    ax.set_title(f'Onda quadrada ({titulo})')
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')

ax2.set_xlim(-0.6, 0.6)
ax2.set_ylim(0.7, 1.4)
ax2.annotate('overshoot (~9% do salto)\nnao desaparece com N grande',
             xy=(0.08, 1.18), xytext=(0.15, 1.3),
             arrowprops=dict(arrowstyle='->', color='black'))
ax1.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

### (d) Espectro de amplitudes

Por fim, vamos visualizar o **espectro** da onda quadrada: os valores de $|a_n|$ e $|b_n|$ em função de $n$. Como a onda quadrada é uma função ímpar (em torno de $x=0$, com a convenção usada aqui), esperamos $a_n \approx 0$ para todo $n$, e $b_n \neq 0$ apenas para $n$ ímpar, decaindo como $1/n$ (decaimento lento, típico de funções descontínuas — é exatamente essa "cauda longa" no espectro que causa o fenômeno de Gibbs).

In [ ]:
N_espectro = 20
ns = np.arange(1, N_espectro + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

largura = 0.4
axes[0].bar(ns - largura / 2, np.abs(a_quad[1:N_espectro + 1]), width=largura,
            color=COR_AJUSTE, label='$|a_n|$')
axes[0].bar(ns + largura / 2, np.abs(b_quad[1:N_espectro + 1]), width=largura,
            color=COR_DADOS, label='$|b_n|$')
axes[0].set_title('Espectro de amplitudes - onda quadrada')
axes[0].set_xlabel('n')
axes[0].set_ylabel('amplitude')
axes[0].legend()

# Em escala log, para evidenciar o decaimento ~1/n dos termos impares
axes[1].semilogy(ns, np.abs(b_quad[1:N_espectro + 1]), 'o-', color=COR_DADOS, label='$|b_n|$ (numerico)')
ns_impares = ns[ns % 2 == 1]
axes[1].semilogy(ns_impares, 4 / (np.pi * ns_impares), 'k--', label=r'$4/(\pi n)$ (referencia, $n$ impar)')
axes[1].set_title('Decaimento do espectro (escala log)')
axes[1].set_xlabel('n')
axes[1].set_ylabel('amplitude')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Vá além: reconstruindo uma onda triangular harmônico a harmônico

Como exercício final (e um pouco lúdico), vamos reconstruir uma **onda triangular** somando harmônicos **um de cada vez**, e ver visualmente como cada novo termo refina a aproximação. Diferente da onda quadrada, a onda triangular é **contínua** (só a derivada é descontínua), então o espectro decai mais rápido ($\sim 1/n^2$) e **não há** fenômeno de Gibbs visível — uma ótima comparação com o caso anterior.

In [ ]:
def onda_triangular(x):
    """Onda triangular, periodo 2L, variando entre -1 e 1,
    obtida como 'integral' (em forma fechada) da onda quadrada.
    """
    x_mod = np.mod(x + L, 2 * L) - L
    return (2 / L) * np.abs(x_mod) - 1


a0_tri, a_tri, b_tri = coeficientes_fourier(onda_triangular, L, N_MAX)

# Como a onda triangular e uma funcao par/impar bem comportada, so os
# harmonicos impares (n = 1, 3, 5, 7, ...) tem amplitude relevante.
Ns_painel = [0, 1, 3, 5, 7, 15]

# Sequencia de reconstrucoes acumulando harmonicos, um painel por valor de N
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
axes = axes.ravel()

for N_acumulado, ax in zip(Ns_painel, axes):
    y_rec = reconstroi_serie(x_rec, a0_tri, a_tri, b_tri, L, N_acumulado)
    ax.plot(x_rec, onda_triangular(x_rec), color='black', linewidth=1.5, alpha=0.5, label='exata')
    ax.plot(x_rec, y_rec, color=COR_AJUSTE, linewidth=2, label=f'N={N_acumulado}')
    ax.set_title(f'ate N = {N_acumulado}')

fig.suptitle('Reconstrucao da onda triangular: refinamento harmonico a harmonico')
plt.tight_layout()
plt.show()

## Conclusão

Neste notebook vimos que o **Método dos Mínimos Quadrados** é, na essência, uma ideia única — **minimizar a soma dos quadrados dos resíduos**, equivalente a uma **projeção ortogonal** no espaço de funções — que se manifesta de formas diferentes conforme a base de funções escolhida:

- Com a **base monomial** $\{1, x, x^2, \dots\}$, obtemos o ajuste polinomial clássico, mas pagamos o preço do mau condicionamento em graus altos;
- Com a **base de Legendre** (ortogonal), o sistema se desacopla e cada coeficiente é obtido por uma projeção simples, de forma numericamente estável;
- Com a **base de senos e cossenos**, obtemos a **Análise Harmônica** (Série de Fourier), fundamental para o estudo de sinais periódicos, vibrações e resposta de estruturas navais a carregamentos oscilatórios (ondas, por exemplo).

Fique atento ao **fenômeno de Gibbs**: ele aparece sempre que há uma descontinuidade na função sendo aproximada, e é um lembrete importante de que mais termos na série nem sempre eliminam todo tipo de erro — alguns erros têm origem na própria natureza da função, não na quantidade de termos usados.